# Preprocessing Pipeline

This notebook performs data cleaning and label reconciliation for the CrisisMMD multimodal dataset. It prepares the annotations for both unimodal and multimodal learning pipelines.

We load the unified annotations (with image hashes), handle noisy/duplicate entries, merge rare classes, and generate fused labels for `mm_info` and `mm_human`.

## Imports and setup

In [ ]:
import os
import pandas as pd

import re
import html
import emoji
import unicodedata

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load paths from .env file
data_dir = os.getenv("DATA_DIR")

# Load the unified TSV file with image hashes
annotations_file = os.path.join(data_dir, "annotations_with_hash.tsv")

In [ ]:
df = pd.read_csv(annotations_file, sep="\t")

In [ ]:
df.head()

## Resolve duplicated tweet entries

Multiple entries may share the same `tweet_id` (i.e., same tweet with multiple images). We reconcile their labels using a custom confidence-aware strategy:

- For each label, we compute a **weighted score** that balances **frequency** and **mean confidence**:
```math
\text{score}(L) = \alpha \cdot \text{frequency}(L) + (1 - \alpha) \cdot \text{average\_confidence}(L)
```
Where $\alpha$ is a tunable parameter:

$\alpha = 1$ → pure frequency

$\alpha = 0$ → pure confidence

- The label with the highest weighted score is selected





In [ ]:
# Show a few examples of duplicated tweet_id entries
print("Sample duplicated tweet_id entries (same tweet, different images):")
sample_tweet_dups = df[df["tweet_id"].duplicated(keep=False)].sort_values("tweet_id").head(10)
display(sample_tweet_dups[["tweet_id", "image_id", "tweet_text", "text_info", "text_info_conf", "text_human", "text_human_conf"]])


In [ ]:
# Mark only the second and subsequent occurrences of each tweet_id as duplicate
df["text_dup"] = df.duplicated(subset=["tweet_id"], keep="first")

In [ ]:
duplicated_tweets = df[df.duplicated(subset=["tweet_id"], keep=False)].copy()

# Find tweet_id groups with inconsistent text_info or text_human
conflicting_tweet_ids = (
	duplicated_tweets.groupby("tweet_id")[["text_info", "text_human"]]
	.nunique()
	.query("text_info > 1 or text_human > 1")
	.index
)

print(f"⚠️ Tweet IDs with conflicting text labels: {len(conflicting_tweet_ids)}")

# Show a few examples
df_conflicting_tweets = df[df["tweet_id"].isin(conflicting_tweet_ids)].sort_values("tweet_id")
display(df_conflicting_tweets[["tweet_id", "tweet_text", "text_info", "text_info_conf", "text_human", "text_human_conf"]].head(10))


In [ ]:
def print_class_distribution(df, label_col):
    print(f"\n`{label_col}` distribution:")
    counts = df[label_col].value_counts(dropna=False)
    percentages = df[label_col].value_counts(normalize=True, dropna=False) * 100
    print(pd.DataFrame({'count': counts, 'percent': percentages.round(2)}))

In [ ]:
print_class_distribution(df, "text_info")
print_class_distribution(df, "text_human")

In [ ]:
def reconcile_with_confidence(group, label_col, conf_col, alpha=0.4):
    """Reconcile a label for a group of duplicated entries using both frequency and confidence."""
    scores = {}
    total = len(group)
    for label in group[label_col].unique():
        label_group = group[group[label_col] == label]
        freq = len(label_group) / total
        avg_conf = label_group[conf_col].mean()
        scores[label] = alpha * freq + (1 - alpha) * avg_conf
    return max(scores, key=scores.get)


In [ ]:
for label_col in ["text_info", "text_human"]:
    conf_col = label_col + "_conf"

    # Reconcile label and corresponding confidence
    reconciled = {}
    for tweet_id, group in duplicated_tweets.groupby("tweet_id"):
        label = reconcile_with_confidence(group, label_col, conf_col, alpha=0.6)
        confidence = group[group[label_col] == label][conf_col].mean()
        reconciled[tweet_id] = (label, confidence)

    # Update label and confidence
    filtered = df["tweet_id"].isin(reconciled.keys())
    df.loc[filtered, label_col] = df.loc[filtered, "tweet_id"].map(lambda tid: reconciled[tid][0])
    df.loc[filtered, conf_col] = df.loc[filtered, "tweet_id"].map(lambda tid: reconciled[tid][1])



In [ ]:
# Tweet text duplication stats
n_total = len(df)
n_dup_tweets = df["text_dup"].sum()
n_unique_conflicting_tweet_ids = len(conflicting_tweet_ids)

print(f"Total duplicated tweet_id rows (excluding first occurrence): {n_dup_tweets}")
print(f"Unique tweet_id groups reconciled: {n_unique_conflicting_tweet_ids}")


In [ ]:
print_class_distribution(df, "text_info")
print_class_distribution(df, "text_human")

## Resolve duplicated images

Some tweets with different IDs may reuse the same image (e.g. retweets). We reconcile the labels for those shared images using the same frequency + confidence method as above.

In [ ]:
# Show a few examples of duplicated images used in different tweets
print("Sample duplicated image hash entries (same image in different tweets):")
sample_img_dups = df[df["img_hash_str"].duplicated(keep=False)].sort_values("img_hash_str").head(10)
display(sample_img_dups[["img_hash_str", "tweet_id", "image_path", "image_info", "image_info_conf", "image_human", "image_human_conf","image_damage", "image_damage_conf"]])

In [ ]:
# Mark only the second and subsequent occurrences of each img_hash_str as duplicate
df["image_dup"] = df.duplicated(subset=["img_hash_str"], keep="first")

In [ ]:
hash_counts = df["img_hash_str"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index

# Find img_hash groups with inconsistent image labels
conflicting_img_hashes = (
	df[df["img_hash_str"].isin(duplicate_hashes)]
	.groupby("img_hash_str")[["image_info", "image_human", "image_damage"]]
	.nunique()
	.query("image_info > 1 or image_human > 1 or image_damage > 1")
	.index
)

print(f"⚠️ Image hashes with conflicting image labels: {len(conflicting_img_hashes)}")

# Show a few examples
df_conflicting_images = df[df["img_hash_str"].isin(conflicting_img_hashes)].sort_values("img_hash_str")
display(df_conflicting_images[["img_hash_str", "tweet_id", "image_path", "image_info", "image_info_conf", "image_human", "image_human_conf","image_damage", "image_damage_conf"]].head(10))


In [ ]:
print_class_distribution(df, "image_info")
print_class_distribution(df, "image_human")
print_class_distribution(df, "image_damage")

In [ ]:
# Count NaN values in "image_damage" and "image_damage_conf"
image_damage_nan = df["image_damage"].isna().sum()
image_damage_conf_nan = df["image_damage_conf"].isna().sum()

print(f"Total NaN values in image_damage: {image_damage_nan}")
print(f"Total NaN values in image_damage_conf: {image_damage_conf_nan}")

In [ ]:
# Fill NaN values in 'image_damage' with "unknown"
df["image_damage"] = df["image_damage"].fillna("unknown")

# Fill NaN values in 'image_damage_conf' with 0.0
df["image_damage_conf"] = df["image_damage_conf"].fillna(0.01)

# Print updated counts
image_damage_nan = df["image_damage"].isna().sum()
unknown_count = (df["image_damage"] == "unknown").sum()
image_damage_conf_nan = df["image_damage_conf"].isna().sum()
zero_conf_count = (df["image_damage_conf"] == 0.01).sum()

print(f"Total NaN values in image_damage: {image_damage_nan}")
print(f"Total 'unknown' values in image_damage: {unknown_count}\n")
print(f"Total NaN values in image_damage_conf: {image_damage_conf_nan}")
print(f"Total 0.01 values in image_damage_conf: {zero_conf_count}\n")


In [ ]:
for label_col in ["image_info", "image_human", "image_damage"]:
    conf_col = label_col + "_conf"

    reconciled = {}
    for img_hash, group in duplicated_tweets.groupby("img_hash_str"):
        label = reconcile_with_confidence(group, label_col, conf_col, alpha=0.6)
        confidence = group[group[label_col] == label][conf_col].mean()
        reconciled[img_hash] = (label, confidence)

    filtered = df["img_hash_str"].isin(reconciled.keys())
    df.loc[filtered, label_col] = df.loc[filtered, "img_hash_str"].map(lambda hid: reconciled[hid][0])
    df.loc[filtered, conf_col] = df.loc[filtered, "img_hash_str"].map(lambda hid: reconciled[hid][1])


In [ ]:
# Image hash duplication stats
n_image_dups = df["image_dup"].sum()  # Only counts second and subsequent occurrences
n_unique_img_hash_groups = len(conflicting_img_hashes)

print(f"Total duplicated image hash rows (excluding first occurrence): {n_image_dups}")
print(f"Unique image hash groups reconciled: {n_unique_img_hash_groups}")

In [ ]:
print_class_distribution(df, "image_info")
print_class_distribution(df, "image_human")
print_class_distribution(df, "image_damage")

## Aggregate low-frequency humanitarian categories

To reduce class imbalance and sparsity in the `*_human` labels, we consolidate the following rare classes:
- `injured_or_dead_people` + `missing_or_found_people` → `affected_individuals`
- `vehicle_damage` → `infrastructure_and_utility_damage`

This improves class distribution and model generalization.

In [ ]:
# Merge small humanitarian categories into broader classes
merge_map = {
    "injured_or_dead_people": "affected_individuals",
    "missing_or_found_people": "affected_individuals",
    "vehicle_damage": "infrastructure_and_utility_damage"
}

df["text_human"] = df["text_human"].replace(merge_map)
df["image_human"] = df["image_human"].replace(merge_map)

In [ ]:
print_class_distribution(df, "text_human")

In [ ]:
print_class_distribution(df, "image_human")

## Multimodal label fusion

We define new target variables that integrate both text and image views:

- `mm_info`: set to `"informative"` if **either** `text_info` or `image_info` is informative
- `mm_human`: we apply strategy based on priority and label confidence scores


In [ ]:
def fuse_info(row):
	if row["text_info"] == "informative" or row["image_info"] == "informative":
		return "informative"
	return "not_informative"

df["mm_info"] = df.apply(fuse_info, axis=1)

In [ ]:
# Priority dictionary
priority = {
    "rescue_volunteering_or_donation_effort": 0.25,
    "affected_individuals": 0.25,
    "infrastructure_and_utility_damage": 0.25,
    "other_relevant_information": 0.15,
    "not_humanitarian": 0.1
}

# Tracking changes
label_changes = []

def fuse_human_confidence_based(row):
    tweet_id = row["tweet_id"]
    img_hash = row["img_hash_str"]
    txt_label = row["text_human"]
    img_label = row["image_human"]
    txt_conf = row["text_human_conf"]
    img_conf = row["image_human_conf"]
    
    # If labels match, keep it
    if txt_label == img_label:
        return txt_label

    # If both are valid but different, use a weighted priority score
    txt_score = priority.get(txt_label, 0) * txt_conf
    img_score = priority.get(img_label, 0) * img_conf

    if txt_score > img_score:
        label_changes.append((row.name, tweet_id, img_hash, txt_label, img_label, "text"))
        return txt_label
    elif img_score > txt_score:
        label_changes.append((row.name, tweet_id, img_hash, txt_label, img_label, "image"))
        return img_label
    else:
        label_changes.append((row.name, tweet_id, img_hash, txt_label, img_label, "conflict"))
        return "conflict"


In [ ]:
df["mm_human"] = df.apply(fuse_human_confidence_based, axis=1)

In [ ]:
changes_df = pd.DataFrame(label_changes, columns=["index", "tweet_id", "img_hash_str", "text_label", "image_label", "chosen"]).drop("index", axis=1)
changes_df

In [ ]:
print_class_distribution(df, "mm_info")
print_class_distribution(df, "mm_human")

## Text Cleaning and Normalization

We apply a set of light preprocessing steps to clean the tweet text while preserving its semantic content. This helps standardize input across samples and reduces noise:

- **Removing URLs, usernames, and hashtags**
- **Unescaping HTML entities** (e.g., `&amp;` → `&`)
- **Stripping emojis and excessive punctuation**
- **Removing accents and diacritics** using Unicode normalization

The goal is to clean the data without losing informative tokens that may be useful for classification (e.g., "rescue", "help", "damage").



In [ ]:
pd.set_option('display.max_colwidth', None)
df["tweet_text"].head(10)

In [ ]:
def preprocess_tweet(text, model_type="embedding"):
	"""
	Preprocess a tweet based on the target model type.

	Parameters:
		text (str): The raw tweet text.
		model_type (str): One of ["embedding", "classic", "bertweet"].
			- "bertweet": minimal cleaning, tailored for BERTweet tokenizer
			- "embedding": general BERT/RoBERTa-style embedding models

	Returns:
		str: Preprocessed tweet text.
	"""
	# Fix text encoding issues
	text = html.unescape(text)
	text = unicodedata.normalize("NFKD", text)

	# Remove leading retweet marker "RT @user:"
	text = re.sub(r'^RT\s+@[\w_]+:\s+', '', text)

	# --- BERTweet-specific preprocessing ---
	if model_type == "bertweet":
		# DO NOT remove URLs, mentions, emojis
		# Just clean up retweet and encoding
		return text.strip()

	# --- For all other models ---
	# Replace URLs with placeholder
	text = re.sub(r'http\S+', '', text)
	
	# Replace mentions with placeholder
	text = re.sub(r'@\w+', '', text)

	# Convert emojis to text (e.g. 😢 → :crying_face:)
	text = emoji.demojize(text, delimiters=(" ", " "))

	# Remove '#' from hashtags, keep the word
	text = re.sub(r'#', '', text)

	# Normalize whitespace
	text = re.sub(r'\s+', ' ', text).strip()

	return text

In [ ]:
# Apply preprocessing and create new columns for each model
df['cleaned_text_bert'] = df['tweet_text'].apply(lambda x: preprocess_tweet(x, model_type='embedding'))
df['cleaned_text_bertweet'] = df['tweet_text'].apply(lambda x: preprocess_tweet(x, model_type='bertweet'))


In [ ]:
df[['tweet_text', 'cleaned_text_bert','cleaned_text_bertweet'] ].head(5)

## Save cleaned dataset

In [ ]:
save_path = os.path.join(data_dir, "preprocessed_annotations.tsv")

df.to_csv(save_path, sep="\t", index=False)